In [1]:
from pyspark.sql import SparkSession

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Tutorial 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/18 10:06:48 WARN Utils: Your hostname, Dinadis-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.13.34.132 instead (on interface en0)
26/09/18 10:06:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/18 10:06:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/18 10:06:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
import pandas as pd
mb_poa_merged = pd.read_csv("../data/raw_abs/mb_poa_merged.csv")

/var/folders/ff/bt51v8c11vxcb8c40b1mpkgh0000gp/T/ipykernel_25074/2797885673.py:2: DtypeWarning: Columns (0,1,3) have mixed types. Specify dtype option on import or set low_memory=False.
  mb_poa_merged = pd.read_csv("../data/raw_abs/mb_poa_merged.csv")


In [3]:
mb_poa_merged.head()

,MB_CODE_2021,SA2_CODE_2021,SA2_NAME_2021,POA_CODE_2021,POA_NAME_2021
0,10000009499,199999499,No usual address (NSW),9494,No usual address (Aust.)
1,10000010000,109011172,Albury - East,2640,2640
2,10000021000,109011176,Lavington,2641,2641
3,10000022000,109011176,Lavington,2641,2641
4,10000023000,109011176,Lavington,2641,2641


In [4]:
# mb_poa_merged is the merged mesh-block-level table with POA_CODE_2021 and SA2_CODE_2021

# Count the number of mesh blocks that falls within the unique post code + SA2 pairs
counts = mb_poa_merged.groupby(['POA_CODE_2021', 'SA2_CODE_2021']).size().reset_index(name='mb_count')

# Sort the df such that the rows are arranged with postcodes sorted in ascending order
# mb_count is sorted from highest to lowest
counts = counts.sort_values(['POA_CODE_2021', 'mb_count'], ascending=[True, False])
print(counts.shape)
print(counts.info())
# if duplicates are present, for example, the rows with the same postcode, keep the first one (the highest mb_count)
# and remove the others 
poa_to_sa2 = counts.drop_duplicates(subset='POA_CODE_2021', keep='first')
print(poa_to_sa2.shape)
print(poa_to_sa2.info())

# Producing the ratio of, in every mesh block of one particular post code, how many of blocks, fall inside a particular SA2? 
poa_totals = mb_poa_merged.groupby('POA_CODE_2021').size().reset_index(name='poa_total')
poa_to_sa2 = poa_to_sa2.merge(poa_totals, on='POA_CODE_2021')
poa_to_sa2['ratio'] = poa_to_sa2['mb_count'] / poa_to_sa2['poa_total']
print(poa_to_sa2.shape)
print(poa_to_sa2.head(10))


(6726, 3)
<class 'pandas.core.frame.DataFrame'>
Index: 6726 entries, 1 to 6725
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   POA_CODE_2021  6726 non-null   object
 1   SA2_CODE_2021  6726 non-null   object
 2   mb_count       6726 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 210.2+ KB
None
(3029, 3)
<class 'pandas.core.frame.DataFrame'>
Index: 3029 entries, 1 to 6725
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   POA_CODE_2021  3029 non-null   object
 1   SA2_CODE_2021  3029 non-null   object
 2   mb_count       3029 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 94.7+ KB
None
(3029, 5)
  POA_CODE_2021 SA2_CODE_2021  mb_count  poa_total     ratio
0          2000     117031644       190        287  0.662021
1          2007     117031646        59         59  1.000000
2          2008     117031639        62   

In [ ]:
# data output directory is
output_relative_dir = '../../data/'
abs_output_dir = output_relative_dir + 'raw_abs'

# save the cleaned POA <-> SA2 mapping df to be reused in the next step 
poa_to_sa2.to_csv(f"{abs_output_dir}/poa_to_sa2.csv", index=False)